## 1. Import libraries and data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_DIR))

from src.clustering import k_distance_elbow_plot

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from kneed import KneeLocator
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

In [2]:
PROJECT_DIR = Path.cwd().resolve().parents[0]

data_path = PROJECT_DIR / "data" / "processed" / "model_table.csv"

df = pd.read_csv(data_path)

FEATURE_COLS = ["Age", "Annual_income_thousands", "Spending_score"]
CATEGORY_COL = ["Gender"]
X = df[FEATURE_COLS]
X.head()

,Age,Annual_income_thousands,Spending_score
0,19,15,39
1,21,15,81
2,20,16,6
3,23,16,77
4,31,17,40


## 2. Standardize data

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. GMM

In [ ]:
k_values = range(2, 11)
cov_types = ["full", "diag"]

rows = []

for k in k_values:
    for cov in cov_types:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=cov, # type: ignore
            random_state=42,
            n_init=5,
        )
        labels = gmm.fit_predict(X_scaled)

        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)
        ch = calinski_harabasz_score(X_scaled, labels)

        rows.append(
            {
                "k": k,
                "cov": cov,
                "silhouette": sil,
                "davies_bouldin": dbi,
                "calinski_harabasz": ch,
                "aic": gmm.aic(X_scaled),
                "bic": gmm.bic(X_scaled),
            }
        )

gmm_metrics = pd.DataFrame(rows).sort_values(by="bic", ascending=True)
gmm_metrics.head(15)

,k,cov,silhouette,davies_bouldin,calinski_harabasz,aic,bic
7,5,diag,0.343103,1.140510,81.440446,1409.593506,1521.736297
9,6,diag,0.408859,0.839181,124.053915,1400.137360,1535.368372
11,7,diag,0.398307,1.076489,113.933241,1386.808450,1545.127683
5,4,diag,0.265701,1.392695,63.260536,1469.398678,1558.453247
13,8,diag,0.339200,1.171015,102.876726,1379.741451,1561.148906
3,3,diag,0.203063,1.909912,55.580909,1496.317873,1562.284220
6,5,full,0.406367,0.935599,116.900931,1415.090385,1576.707936
4,4,full,0.260409,1.398142,62.578537,1452.738951,1581.373328
2,3,full,0.204944,1.905337,55.651157,1486.324654,1581.975857
15,9,diag,0.366983,0.888826,111.597423,1390.049052,1594.544729


In [ ]:
def best_gmm_config(
    gmm_grid: pd.DataFrame
) -> pd.Series:
    df = gmm_metrics.copy()

    df = df.dropna(subset=["silhouette", "davies_bouldin", "calinski_harabasz"])

    if df.empty:
        raise ValueError(
            "No DBSCAN configs left after filtering. "
            "Change max_noise or eps/min_samples."
        )

    # rank
    df = df.sort_values(by="bic", ascending=True)
    return df.iloc[0]

best_cfg = best_gmm_config(gmm_metrics)
best_cfg

k                              5
cov                         diag
silhouette              0.343103
davies_bouldin           1.14051
calinski_harabasz      81.440446
aic                  1409.593506
bic                  1521.736297
Name: 7, dtype: object

## 3.1 Rerun model using best parameters

In [13]:
gmm = GaussianMixture(
    n_components=int(best_cfg["k"]),
    covariance_type=best_cfg["cov"],
    random_state=42,
    n_init=5,
)
        
labels = gmm.fit_predict(X_scaled)

gmm_centers_scaled = gmm.means_
gmm_centers_original = scaler.inverse_transform(gmm_centers_scaled)

gmm_centers_df = pd.DataFrame(
    gmm_centers_original,
    columns=FEATURE_COLS
)

sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)

## 3.2 Plot clusters

In [10]:
import plotly.graph_objects as go
import plotly.express as px

df_original = df[FEATURE_COLS].to_numpy()

# plot dataframe
plot_df = pd.DataFrame(df_original, columns=FEATURE_COLS).assign(
    cluster=labels.astype(str)
)

# points
fig = px.scatter_3d(
    plot_df,
    x=FEATURE_COLS[0],
    y=FEATURE_COLS[1],
    z=FEATURE_COLS[2],
    color="cluster",
    opacity=0.85,
)

# centroids
fig.add_trace(go.Scatter3d(
    x=gmm_centers_original[:, 0],
    y=gmm_centers_original[:, 1],
    z=gmm_centers_original[:, 2],
    mode="markers",
    marker=dict(
        size=8,
        symbol="x",
        color="red",
        line=dict(width=2, color="black")
    ),
    name="Centroids"
))

fig.update_layout(
    title="GMM clusters with centroids",
    scene=dict(
        xaxis_title=FEATURE_COLS[0],
        yaxis_title=FEATURE_COLS[1],
        zaxis_title=FEATURE_COLS[2],
    ),
    legend=dict(
        title="Cluster",
        itemsizing="constant",
        x=1.15,
        y=1,
        xanchor="left",
        yanchor="top"
    ),
    margin=dict(r=180)
)

fig.show()
fig.write_html(f"{PROJECT_DIR}/reports/figures/GMM_clusters_3d.html", include_plotlyjs="cdn")

## 3.3 Cluster summary

In [12]:
labels = gmm.fit_predict(X_scaled)

gmm_model = X.copy()
gmm_model["cluster"] = labels

summary = (
    gmm_model
    .groupby("cluster")[FEATURE_COLS]
    .agg(["count", "mean", "median", "std"])
)
summary

Age                              Annual_income_thousands             \
        count       mean median        std                   count       mean   
cluster                                                                         
0          23  45.217391   46.0  13.228607                      23  26.304348   
1          22  25.272727   23.5   5.257030                      22  25.727273   
2          39  32.692308   32.0   3.728650                      39  86.538462   
3          38  40.394737   41.5  11.376931                      38  87.000000   
4          78  43.128205   47.0  16.553227                      78  54.615385   

                          Spending_score                               
        median        std          count       mean median        std  
cluster                                                                
0         25.0   7.893811             23  20.913043   17.0  13.017167  
1         24.5   7.566731             22  79.363636   77.0  10.504174  
2         79.0  16.312485             39  82.128205   83.0   9.364489  
3         80.0  16.271348             38  18.631579   16.5  10.915947  
4         54.0   8.430358             78  50.025641   50.0   6.083775

### Clusters found:
0 - Pre-retirement individuals, low earners and low spenders (Age ~45, Income ~26k, Score ~20)  
1 - Young professionals, low earners and high spenders (Age ~25, Income ~25k, Score ~79)  
2 - Young professionals, high earners and high spenders (Age ~32, Income ~86k, Score ~82)    
3 - Mid-career adults, high earners and low spenders  (Age ~40, Income ~87k, Score ~18)  
4 - Mid-career adults, middle earners and middle spenders  (Age ~43, Income ~54k, Score ~50)  